In [67]:
# Import necessary libraries
import os
import psycopg2
import pandas as pd
from psycopg2 import sql
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

load_dotenv()  # Load environment variables from .env file

True

### Load the raw datasets 

In [68]:
# Load each raw CSV file into a Pandas DataFrame
patients = pd.read_csv("../data/raw_data/patients.csv")
appointments = pd.read_csv("../data/raw_data/appointments.csv")
billing = pd.read_csv("../data/raw_data/billing.csv")
doctors = pd.read_csv("../data/raw_data/doctors.csv")

In [69]:
patients

,patient_id,full_name,gender,age,phone_number,email,city,registration_date
0,P01600,Mary Nwosu,M,12,0808-783-0713,not_an_email,Kano,24/04/2023
1,P00162,Grace Ogunleye,NaN,25,0802 815 3780,NaN,Benin city,"Aug 28, 2024"
2,P00986,David Garba,NaN,15,NaN,david.garba474@email.com,Lagos,2024/09/06
3,P00503,GRACE NWOSU,female,-5,0809-542-1859,NaN,ABUJA,2022-08-24
4,P00463,Tolu Ojo,NaN,NaN,0800-757-8203,tolu.ojo372@email.com,Benin city,26/04/2026
...,...,...,...,...,...,...,...,...
1665,P01559,Grace Balogun,female,46,8053985671,grace.balogun38@yahoo.com,IBADAN,2024-10-08
1666,P01184,Amaka Sani,Female,28,+2348010265670,AMAKA.SANI916@GMAIL.COM,lagos,2022/09/09
1667,P00494,Tunde Usman,female,Thirty-Five,0804 029 4698,tunde.usman30@email.com,Port Harcourt,2021-07-15
1668,P00528,Hannah Johnson,female,Thirty-Five,08077024790,hannah.johnson285@yahoo.com,ibadan,2024-05-15


In [70]:
### Explore each DataFrame to understand the structure and content of the data
# patients.tail()  # Display the last few rows of the patients DataFrame
# appointments.head()  # Display the first few rows of the appointments DataFrame
# billing.info()  # Display summary information about the billing DataFrame
# doctors.shape  # Display the shape of the doctors DataFrame
# patients.describe()  # Display summary statistics of the patients DataFrame
# appointments.isnull().sum()  # Check for missing values in the appointments DataFrame
# patients.duplicated().sum()  # Check for duplicate rows in the patients DataFrame

## Data Cleaning

In [71]:
# Removing duplicates from the patients dataset
patients = patients.drop_duplicates()

In [72]:
# Standardizing the city names in the patients dataset 
patients["city"] = patients["city"].str.title()


In [73]:
# Standardize gender values in the patients dataset
patients["gender"] = patients["gender"].replace({
    "M":"Male",
    "F":"Female",
    "male":"Male",
    "female":"Female"
})

In [74]:
# Filling missing gender nan
patients["gender"] = patients["gender"].fillna("Unknown")

In [75]:
# Replacing missing emails with a placeholder value
patients["email"] = patients["email"].fillna("Not Provided")

In [76]:
# Converting the age values to integers/numeric 
patients["age"] = pd.to_numeric(
    patients["age"], 
    errors="coerce" #Replace anything that cannot be converted to a number with NaN(Not a Number). Invalid age become NaN
)

In [77]:
# Replace missing age values 
patients["age"] = patients["age"].fillna(
    patients["age"].median()
)

In [78]:
# Replace negative ages with missing values
patients.loc[patients["age"] < 0, "age"] = pd.NA

# Fill missing values with the median
patients["age"] = patients["age"].fillna(
    patients["age"].median()
)

# Round ages
patients["age"] = patients["age"].round()

# Convert to integers
patients["age"] = patients["age"].astype(int)

In [79]:
# Find missing IDs
missing_rows = patients[patients["patient_id"].isna()].index

# Generate new IDs
for i, idx in enumerate(missing_rows, start=1601):
    patients.loc[idx, "patient_id"] = f"P{i:05d}"

In [80]:
# Standardize names to Title Case
patients["full_name"] = patients["full_name"].str.title() 

In [81]:
# Fill the missing values in the phone_number column with "Unknown"
patients["phone_number"] = patients["phone_number"].fillna("Unknown")

In [82]:
# Remove spaces from phone numbers so they use one consistent format
patients["phone_number"] = patients["phone_number"].str.replace(" ", "", regex=False)

In [83]:
# Remove hyphens from phone numbers
patients["phone_number"] = patients["phone_number"].str.replace("-", "", regex=False)

In [84]:
# Change Nigeria's +234 country code to a leading zero
""" 
Skipping the first four characters of the phone number (which is +234) and adding a leading zero to the remaining part of the phone number.
"""
patients["phone_number"] = patients["phone_number"].apply(
    lambda x: "0" + x[4:] if x.startswith("+234") else x
)

In [85]:
# Add a leading zero to local phone numbers that contain 10 characters
patients["phone_number"] = patients["phone_number"].apply(
    lambda phone: "0" + phone if len(phone) == 10 else phone
)

In [86]:
# Replace a known invalid placeholder with the standard value, Unknown
patients["phone_number"] = patients["phone_number"].replace({
    "not_available": "Unknown"
})

In [87]:
# Replace invalid phone numbers
# A Nigerian phone number should contain 11 digits.
# Anything shorter or longer is invalid.

"""
This will automatically convert:
12345 into Unknown

while leaving 08026951234 unchanged.
"""

patients["phone_number"] = patients["phone_number"].apply(
    lambda phone: phone if phone == "Unknown" or len(phone) == 11 else "Unknown"
)


In [88]:
# Convert email addresses to lowercase for consistent matching
patients["email"] = patients["email"].str.lower()

In [89]:
# Standardize common missing or invalid email labels as Unknown
patients["email"] = patients["email"].replace({
    "Not Provided": "Unknown",
    "not_an_email": "Unknown",
    "not provided": "Unknown",
    "not available": "Unknown",
    "n/a": "Unknown",
    "none": "Unknown"
})

In [90]:
# Instead of automatically fixing them (which may introduce incorrect data), mark them as unknown.
patients["email"] = patients["email"].apply(
    lambda email: email if email == "Unknown" or "@" in email else "Unknown"
)

### Date Transformation 

In [91]:
# Parse the mixed source formats and keep valid registration dates as date values
patients["registration_date"] = (
    pd.to_datetime(
        patients["registration_date"],
        format="mixed",
        errors="coerce"
    )
    .dt.date
)

In [92]:
# Keep unparseable registration dates as database-compatible NULL values
patients["registration_date"] = patients["registration_date"].where(
    patients["registration_date"].notna(), None
)

In [93]:
# Creating the Age group column 
patients["age_group"] = pd.cut(
    patients["age"],
    bins=[0,17,35,60,100],
    labels=[
        "Child",
        "Young Adult",
        "Adult",
        "Senior"
    ]
)

In [94]:
# Find the largest appointment_id
# Get the highest appointment number
max_id = (
    appointments["appointment_id"]
    .dropna()
    .str.replace("A", "", regex=False)
    .astype(int)
    .max()
)

In [95]:
# Generate new appointment IDs for missing values
# Find rows with missing IDs
missing_rows = appointments["appointment_id"].isna()

# Create new IDs
new_ids = [
    f"A{num:06d}"
    for num in range(max_id + 1, max_id + 1 + missing_rows.sum())
]

# Assign them
appointments.loc[missing_rows, "appointment_id"] = new_ids

In [96]:
# Keep appointments with an unknown patient, but store the invalid reference as NULL
appointments["patient_id"] = appointments["patient_id"].where(
    appointments["patient_id"].isin(patients["patient_id"]), pd.NA
)

In [97]:
# Keep appointments with an unknown doctor, but store the invalid reference as NULL
appointments["doctor_id"] = appointments["doctor_id"].where(
    appointments["doctor_id"].isin(doctors["doctor_id"]), pd.NA
)

In [98]:
# Remove surrounding spaces before standardizing appointment statuses
appointments["appointment_status"] = appointments["appointment_status"].str.strip()

In [99]:
# Standardize capitalization, then merge both No-show spellings into No Show
appointments["appointment_status"] = (
    appointments["appointment_status"]
    .str.title()
    .replace({"No-Show": "No Show"})
) 

In [100]:
# Fill up the 345 missing values in the appointment_status column with "Pending"
appointments["appointment_status"] = appointments["appointment_status"].fillna("Pending")

In [101]:
# Convert the appointment_date column to datetime format first
appointments["appointment_date"] = pd.to_datetime(
    appointments["appointment_date"],
    errors="coerce",
    format="mixed"
)

In [102]:
# Standardize the format. We do not need to convert the appointment_date column to date only because it is already in the correct format. We will just standardize the format to YYYY-MM-DD.
appointments["appointment_date"] = (
    appointments["appointment_date"]
    .dt.strftime("%Y-%m-%d")
)


In [103]:
# Fill missing visit descriptions with a consistent placeholder
appointments["reason_for_visit"] = appointments["reason_for_visit"].fillna("Unknown")

In [104]:
# remove rows with missing invoice IDs since they represent the primary key.
billing = billing.dropna(subset=["invoice_id"])

In [105]:
# Billing rows require a valid patient because patient_id is a mandatory foreign key
billing = billing[billing["patient_id"].isin(patients["patient_id"])].copy()

In [106]:
# Parse mixed invoice-date formats and store valid dates as YYYY-MM-DD
billing["invoice_date"] = pd.to_datetime(
    billing["invoice_date"],
    errors="coerce",
    format="mixed"
).dt.strftime("%Y-%m-%d")

In [107]:
# Convert to Numeric  
billing["amount_charged"] = pd.to_numeric(
    billing["amount_charged"],
    errors="coerce"
)

In [108]:
# Replace negative amounts with missing values
billing.loc[
    billing["amount_charged"] < 0,
    "amount_charged"
] = pd.NA

In [109]:
# Fill missing values with median 
billing["amount_charged"] = billing["amount_charged"].fillna(
    billing["amount_charged"].median()
)

In [110]:
# Round the amount_charged to the nearest whole number
billing["amount_charged"] = billing["amount_charged"].round().astype(int)

In [111]:
# Convert to numeric first 
billing["amount_paid"] = pd.to_numeric(
    billing["amount_paid"],
    errors="coerce"
)


In [112]:
# replace negative values if any 
billing.loc[
    billing["amount_paid"] < 0,
    "amount_paid"
] = 0



In [113]:
# fill missing values with 0
billing["amount_paid"] = billing["amount_paid"].fillna(0)

In [114]:
# Round to nearest whole number
billing["amount_paid"] = billing["amount_paid"].round().astype(int)

In [115]:
# Convert to title case for the payment_method column 
billing["payment_method"] = (
    billing["payment_method"]
    .str.strip()
    .str.title()
)  

In [116]:
# Replace missing values with unknown 
billing["payment_method"] = billing["payment_method"].fillna("Unknown")

In [117]:
# Convert to title case 
billing["payment_status"] = (
    billing["payment_status"]
    .str.strip()
    .str.title()
)

In [118]:
# Fill missing values 
billing["payment_status"] = billing["payment_status"].fillna("Unknown")

In [119]:
# Get the highest existing doctor number
max_id = (
    doctors["doctor_id"]
    .dropna()
    .str.replace("D", "", regex=False)
    .astype(int)
    .max()
)

# Find rows with missing doctor_id
missing_rows = doctors["doctor_id"].isna()

# Generate new doctor IDs
new_ids = [
    f"D{num:04d}"
    for num in range(max_id + 1, max_id + 1 + missing_rows.sum())
]

# Assign the new IDs
doctors.loc[missing_rows, "doctor_id"] = new_ids

In [120]:
# Remove extra spaces and standardize doctor names to title case
doctors["doctor_name"] = (
    doctors["doctor_name"]
    .str.strip()
    .str.title()
)

In [121]:
# Remove extra spaces and standardize specialty names to title case
doctors["specialty"] = (
    doctors["specialty"]
    .str.strip()
    .str.title()
)

In [122]:
# Fill the missing values in the specialty with unknown
doctors["specialty"] = doctors["specialty"].fillna("Unknown")

In [123]:
# Standardize Capitalization 
doctors["branch"] = (
    doctors["branch"]
    .str.strip()
    .str.title()
)

In [124]:
# Fill Missing Values in the branch column with "Unknown" 
doctors["branch"] = doctors["branch"].fillna("Unknown")

### Validate and export the cleaned DataFrames

In [125]:
# Fail early if a primary key or relationship would violate the database schema
primary_keys = {
    "patients": (patients, "patient_id"),
    "doctors": (doctors, "doctor_id"),
    "appointments": (appointments, "appointment_id"),
    "billing": (billing, "invoice_id"),
}

for table_name, (dataframe, key) in primary_keys.items():
    assert dataframe[key].notna().all(), f"{table_name}.{key} contains NULL values"
    assert dataframe[key].is_unique, f"{table_name}.{key} contains duplicates"

assert appointments["patient_id"].dropna().isin(patients["patient_id"]).all()
assert appointments["doctor_id"].dropna().isin(doctors["doctor_id"]).all()
assert billing["patient_id"].isin(patients["patient_id"]).all()
assert billing["amount_charged"].ge(0).all()
assert billing["amount_paid"].ge(0).all()
allowed_appointment_statuses = {
    "Pending", "Cancelled", "Completed", "Scheduled", "Rescheduled", "No Show"
}
assert appointments["appointment_status"].isin(allowed_appointment_statuses).all(), (
    "appointments contains an unsupported appointment_status"
)

print("Cleaned datasets passed primary-key, foreign-key, and amount checks.")

Cleaned datasets passed primary-key, foreign-key, and amount checks.


In [126]:
# Save each cleaned DataFrame as a CSV without the Pandas row index
# Patients cleaned data
patients.to_csv("../data/cleaned_data/patients_cleaned.csv", index=False) 
# Appointments cleaned_data
appointments.to_csv("../data/cleaned_data/appointments_cleaned.csv", index=False)
# Billing cleaned_data
billing.to_csv("../data/cleaned_data/billing_cleaned.csv", index=False)
# Doctors cleaned_data
doctors.to_csv("../data/cleaned_data/doctors_cleaned.csv", index=False)

In [127]:
# Read the saved CSV files back into DataFrames for database loading
patients_cleaned = pd.read_csv("../data/cleaned_data/patients_cleaned.csv")
appointments_cleaned = pd.read_csv("../data/cleaned_data/appointments_cleaned.csv")
billing_cleaned = pd.read_csv("../data/cleaned_data/billing_cleaned.csv")
doctors_cleaned = pd.read_csv("../data/cleaned_data/doctors_cleaned.csv")

### Create database and connect to the database

In [128]:
# Read the PostgreSQL connection settings from the .env file
db_name = os.getenv("DB_NAME")
db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_host = os.getenv("DB_HOST")
db_port = os.getenv("DB_PORT", "5432")

# Connect to PostgreSQL's maintenance database so the target database can be created
connection = psycopg2.connect(
    dbname=os.getenv("DB_MAINTENANCE_NAME", "postgres"),
    user=db_user,
    password=db_password,
    host=db_host,
    port=db_port,
)

# CREATE DATABASE must run outside a transaction, so enable autocommit
connection.autocommit = True
# Create a cursor for running SQL commands through psycopg2
cursor = connection.cursor()

# Check whether the database exists
cursor.execute(
    "SELECT 1 FROM pg_database WHERE datname = %s",
    (db_name,),
)

# Create it when it does not exist
if cursor.fetchone() is None:
    cursor.execute(sql.SQL("CREATE DATABASE {}").format(sql.Identifier(db_name)))
    print(f"Database '{db_name}' created.")
else:
    print(f"Database '{db_name}' already exists.")

# Close the temporary cursor and connection to release resources
cursor.close()
connection.close()

# Connect SQLAlchemy to the new database
database_url = URL.create(
    drivername="postgresql+psycopg2",
    username=db_user,
    password=db_password,
    host=db_host,
    port=int(db_port),
    database=db_name,
)

# Create the reusable SQLAlchemy engine used by Pandas to_sql and read_sql
engine = create_engine(database_url)

Database 'schemas_db' already exists.


In [129]:
appointments["appointment_status"].unique()

<StringArray>
['Pending', 'Cancelled', 'Completed', 'Scheduled', 'Rescheduled', 'No Show']
Length: 6, dtype: str

In [130]:
# Create the schema and constrained tables in dependency order.
# The source identifiers include prefixes (P, D, A, and INV), so they are stored as TEXT.
create_table_query = """
CREATE SCHEMA IF NOT EXISTS healthplus;

CREATE TABLE IF NOT EXISTS healthplus.patients (
    patient_id TEXT PRIMARY KEY,
    full_name TEXT NOT NULL,
    gender TEXT NOT NULL,
    age INTEGER NOT NULL CHECK (age >= 0),
    phone_number TEXT NOT NULL,
    email TEXT NOT NULL,
    city TEXT NOT NULL,
    registration_date DATE,
    age_group TEXT
);

CREATE TABLE IF NOT EXISTS healthplus.doctors (
    doctor_id TEXT PRIMARY KEY,
    doctor_name TEXT NOT NULL,
    specialty TEXT NOT NULL,
    branch TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS healthplus.appointments (
    appointment_id TEXT PRIMARY KEY,
    patient_id TEXT,
    doctor_id TEXT,
    appointment_date DATE,
    appointment_status TEXT NOT NULL DEFAULT 'Pending' CHECK (
        appointment_status IN (
            'Pending', 'Cancelled', 'Completed', 'Scheduled', 'Rescheduled', 'No Show'
        )
    ),
    reason_for_visit TEXT NOT NULL,
    FOREIGN KEY (patient_id) REFERENCES healthplus.patients(patient_id),
    FOREIGN KEY (doctor_id) REFERENCES healthplus.doctors(doctor_id)
);

CREATE TABLE IF NOT EXISTS healthplus.billing (
    invoice_id TEXT PRIMARY KEY,
    patient_id TEXT NOT NULL,
    invoice_date DATE,
    amount_charged NUMERIC(12, 2) NOT NULL CHECK (amount_charged >= 0),
    amount_paid NUMERIC(12, 2) NOT NULL DEFAULT 0 CHECK (amount_paid >= 0),
    payment_method TEXT NOT NULL,
    payment_status TEXT NOT NULL,
    FOREIGN KEY (patient_id) REFERENCES healthplus.patients(patient_id)
);
"""

with engine.begin() as connection:
    connection.exec_driver_sql(create_table_query)

print("Schema and tables are ready.")

Schema and tables are ready.


### Load the Cleaned Data

In [131]:
# Load the cleaned CSV files that will become PostgreSQL tables
patients_cleaned = pd.read_csv("../data/cleaned_data/patients_cleaned.csv")
appointments_cleaned = pd.read_csv("../data/cleaned_data/appointments_cleaned.csv")
billing_cleaned = pd.read_csv("../data/cleaned_data/billing_cleaned.csv")
doctors_cleaned = pd.read_csv("../data/cleaned_data/doctors_cleaned.csv")

In [132]:
# Reload all four tables without dropping their keys, checks, or relationships.
# Parent tables are loaded before the child tables that reference them.
with engine.begin() as connection:
    connection.exec_driver_sql(
        "TRUNCATE TABLE healthplus.billing, healthplus.appointments, "
        "healthplus.doctors, healthplus.patients"
    )

    patients_cleaned.to_sql(
        "patients", connection, schema="healthplus", if_exists="append", index=False
    )
    doctors_cleaned.to_sql(
        "doctors", connection, schema="healthplus", if_exists="append", index=False
    )
    appointments_cleaned.to_sql(
        "appointments", connection, schema="healthplus", if_exists="append", index=False
    )
    billing_cleaned.to_sql(
        "billing", connection, schema="healthplus", if_exists="append", index=False
    )

print("Cleaned data loaded into the healthplus schema.")

Cleaned data loaded into the healthplus schema.
